# 🟦 Pattern 3 — Planning Agents

> **One-line definition:** write the whole plan **first**, then execute it step by step,
> re-planning only when reality disagrees.

Also known as **Plan-and-Execute** (Wang et al., "Plan-and-Solve", 2023).

---

## 1. Mental Model

```
        Goal
          │
          ▼
      ┌────────┐
      │  PLAN  │   1 big LLM call -> ordered list of steps
      └────────┘
          │
          ▼
      ┌────────┐
   ┌─►│ EXECUTE│   run step[0] (small model + tools)
   │  └────────┘
   │      │
   │      ▼
   │  ┌────────┐
   └──│ REPLAN │   done? -> answer   |   not done? -> revise remaining steps
      └────────┘
          │
          ▼
       Answer
```

---

## 2. ReAct vs Planning — the core difference

| | ReAct | Planning |
|---|---|---|
| When is the path decided? | **during** execution, one step at a time | **before** execution, all at once |
| Full task visible to LLM? | only implicitly, via history | ✅ explicitly, as a list |
| Can a human review the plan? | ❌ no | ✅ yes (great for HITL) |
| Long-horizon drift | 🔴 high — forgets the goal | 🟢 low — the plan anchors it |
| Cost per step | 1 big LLM call each step | 1 big call to plan + cheap calls to execute |
| Adapts to surprises | 🟢 instantly | 🟡 only at the re-plan point |

👉 **Use planning when the task has many steps and the agent tends to lose the thread.**

---

## 3. Key Properties (pointwise)

| Property | Planning agent |
|---|---|
| Planning | ✅ explicit, upfront, inspectable |
| Loop | ✅ execute ⇄ replan |
| Memory | 🟡 the plan + past_steps list |
| Reflection | 🟡 partial — replan is a weak critic |
| Termination | plan is empty **or** replanner emits a final answer |
| Cost | 💰 medium — big planner, cheap executors |
| Predictability | 🟢 high — you can read the plan before it runs |

---

## 4. The confusing parts (resolved 👇)

### Q1: "Isn't re-planning just ReAct with extra steps?"

No. The distinction is **what the LLM is asked to produce**.

- **ReAct**: "given everything so far, what's the *next action*?" → myopic
- **Replan**: "given the original goal, the *full remaining plan*, and what just happened —
  is the plan still correct?" → holistic

The replanner sees the **goal** and the **whole remaining plan**. The ReAct agent only ever
sees a message list and has to re-derive intent every turn.

---

### Q2: "Why is `Response` a separate schema from `Plan`?"

Because the replanner must be able to say **two different things**:

| Says | Meaning | Graph goes to |
|---|---|---|
| `Plan(steps=[...])` | "not done, here's what's left" | `execute` |
| `Response(response="...")` | "done, here's the answer" | `END` |

We model this as a **union type** (`Act`), and the router just checks which one came back.
This is cleaner than parsing "is the plan empty?" from text.

---

### Q3: "Why does the plan shrink instead of tracking an index?"

Both work. **Shrinking is safer** because:
- If the replanner *reorders* or *inserts* steps, an index would point at the wrong thing.
- "Plan is empty" is an unambiguous termination signal.
- The state stays self-describing — you can print it and see exactly what's left.

⚠️ The trap: the replanner must return **only the remaining steps**, never the completed
ones. Say this explicitly in the prompt or you get an infinite loop.

---

### Q4: "Executor uses a ReAct agent inside the Planning agent — isn't that mixing patterns?"

Yes, **on purpose**. This is the key insight of the whole taxonomy:

```
Planning agent = Planner (LLM)  +  Executor (a ReAct agent)  +  Replanner (LLM)
```

Patterns **compose**. The planner handles the long horizon; the ReAct executor handles the
messy tool work inside a single step. Neither could do the whole job alone.

---

## 5. Graph we will build

```
   START
     │
     ▼
 ┌────────┐
 │ planner│  ── Plan(steps=[s1, s2, s3])
 └────────┘
     │
     ▼
 ┌────────┐
 │executor│  ── runs steps[0] via a ReAct sub-agent
 └────────┘  ── appends (step, result) to past_steps
     │
     ▼
 ┌────────┐
 │replanner│ ── returns Plan(remaining) OR Response(final)
 └────────┘
     │
     ├── Plan     ──► executor   (loop)
     └── Response ──► END
```


## 0. Setup

**Install once:**

```bash
pip install langgraph langchain-openai langchain-core
```

**Set your API key** (any chat model works — swap the import if you use Anthropic/Ollama).


In [ ]:
# --- Standard setup used by every notebook in this series ---
import os, getpass

def _set(var: str):
    """Prompt for a key only if it's not already in the environment."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set("GROQ_API_KEY")

from langchain_groq import ChatGroq

# temperature=0 -> deterministic-ish output, easier to reason about while learning
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("LLM ready")

---

## 6. Step 1 — Tools + the executor sub-agent

The executor is deliberately **a ReAct agent**. It gets one step at a time and doesn't
need to know the big picture.


In [ ]:
from langchain_core.tools import tool


@tool
def search_flights(origin: str, destination: str) -> str:
    """Search available flights between two cities."""
    return f"Flights {origin}->{destination}: Eurowings 07:20 (€89), Lufthansa 14:05 (€142)"


@tool
def search_hotels(city: str, nights: int) -> str:
    """Find hotels in a city for a given number of nights."""
    return f"{city} ({nights} nights): Hotel Alpin €95/night, Stadt Hotel €140/night"


@tool
def get_attractions(city: str) -> str:
    """List the top tourist attractions in a city."""
    data = {
        "salzburg": "Hohensalzburg Fortress, Mirabell Gardens, Mozart's Birthplace",
        "vienna":   "Schönbrunn Palace, Stephansdom, Prater, Belvedere",
    }
    return data.get(city.lower(), f"No attraction data for {city}")


@tool
def calculate_budget(items: str) -> str:
    """Sum a comma-separated list of numeric costs, e.g. '89, 95, 95, 40'."""
    try:
        nums = [float(x.strip()) for x in items.split(",")]
        return f"Total: €{sum(nums):.2f}"
    except Exception as e:
        return f"Error: {e}"


planning_tools = [search_flights, search_hotels, get_attractions, calculate_budget]

In [ ]:
from langgraph.prebuilt import create_react_agent

# The EXECUTOR. It is a full ReAct agent, but scoped to ONE step at a time.
# This is pattern composition: Planning wraps ReAct.
executor_agent = create_react_agent(
    llm,
    tools=planning_tools,
    prompt="You execute exactly ONE task. Use tools. Be concise and factual.",
)
print("Executor (ReAct sub-agent) ready")

---

## 7. Step 2 — Schemas

**Why Pydantic and not free text?** The router needs to distinguish "keep going" from
"we're done" reliably. Structured output makes that a type check, not a regex.


In [ ]:
from typing import List, Union, Annotated, Tuple, TypedDict
import operator
from pydantic import BaseModel, Field


class Plan(BaseModel):
    """An ordered list of steps to accomplish the goal."""
    steps: List[str] = Field(description="Ordered steps. Each must be self-contained and independently executable.")


class Response(BaseModel):
    """The final answer to the user. Emitting this ENDS the graph."""
    response: str


class Act(BaseModel):
    """Union type. The replanner returns exactly one of these.

    Plan     -> more work remains, here are the REMAINING steps
    Response -> we're finished, here is the answer
    """
    action: Union[Response, Plan] = Field(
        description="Use Response if you can answer the user now. "
                    "Use Plan if more steps are still required."
    )

### The State — and why `past_steps` needs a reducer

| Key | Reducer? | Why |
|---|---|---|
| `input` | ❌ | set once, never changes |
| `plan` | ❌ | replanner **replaces** it wholesale |
| `past_steps` | ✅ `operator.add` | must **accumulate** across iterations |
| `response` | ❌ | written once at the end |

`Annotated[list, operator.add]` is the generic "append instead of overwrite" reducer —
`add_messages` is just a smarter, message-aware version of the same idea.


In [ ]:
class PlanState(TypedDict):
    input: str                                            # the original goal
    plan: List[str]                                       # remaining steps (shrinks)
    past_steps: Annotated[List[Tuple[str, str]], operator.add]   # (step, result) — ACCUMULATES
    response: str                                         # final answer

---

## 8. Step 3 — The planner node


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "For the given objective, produce a simple step-by-step plan.\n"
     "RULES:\n"
     "- Each step must be a self-contained task with all info needed to execute it.\n"
     "- Do not add fluff steps. The final step must yield the final answer.\n"
     "- Do not skip steps; the result of the last step is the answer."),
    ("user", "{input}"),
])

# Chain: prompt -> LLM constrained to the Plan schema
planner = planner_prompt | llm.with_structured_output(Plan)


def plan_node(state: PlanState) -> dict:
    """ONE upfront LLM call that decomposes the whole goal."""
    plan = planner.invoke({"input": state["input"]})
    print("\n📋 PLAN:")
    for i, s in enumerate(plan.steps, 1):
        print(f"   {i}. {s}")
    return {"plan": plan.steps}

---

## 9. Step 4 — The executor node

**Key detail:** it executes `plan[0]` only, and records the result. It does **not**
remove the step from the plan — the *replanner* decides what's left. This keeps a single
source of truth.


In [ ]:
def execute_node(state: PlanState) -> dict:
    """Run ONE step (plan[0]) using the ReAct sub-agent."""
    plan = state["plan"]
    task = plan[0]

    # Give the executor the full plan as context, but tell it to do only step 1.
    plan_text = "\n".join(f"{i}. {s}" for i, s in enumerate(plan, 1))
    prompt = (
        f"Overall plan:\n{plan_text}\n\n"
        f"You are executing ONLY step 1: {task}"
    )

    print(f"\n⚙️  EXECUTING: {task}")
    result = executor_agent.invoke({"messages": [("user", prompt)]})
    answer = result["messages"][-1].content
    print(f"   ✓ {answer[:150]}")

    # operator.add reducer APPENDS this tuple to past_steps.
    return {"past_steps": [(task, answer)]}

---

## 10. Step 5 — The replanner node

This is where planning earns its keep. It sees:
1. the **original goal**,
2. the **original plan**,
3. everything **already done**.

…and decides: revise, or finish.


In [ ]:
replanner_prompt = ChatPromptTemplate.from_template(
    """For the given objective, come up with a simple step-by-step plan.
Each step must be self-contained. Do not add fluff steps.

Your objective was:
{input}

Your original plan was:
{plan}

You have currently completed these steps:
{past_steps}

Update the plan accordingly.
⚠️ ONLY include steps that STILL NEED TO BE DONE. Do not repeat completed steps.
If no more steps are needed and you can answer the user, respond with a Response instead."""
)

replanner = replanner_prompt | llm.with_structured_output(Act)


def replan_node(state: PlanState) -> dict:
    """Decide: revise the remaining plan, or emit the final answer."""
    output = replanner.invoke({
        "input": state["input"],
        "plan": "\n".join(f"- {s}" for s in state["plan"]),
        "past_steps": "\n".join(f"- {s}: {r[:200]}" for s, r in state["past_steps"]),
    })

    # The union type makes routing a simple isinstance check.
    if isinstance(output.action, Response):
        print("\n✅ REPLANNER: done")
        return {"response": output.action.response}

    print(f"\n🔄 REPLANNER: {len(output.action.steps)} step(s) remaining")
    return {"plan": output.action.steps}     # no reducer -> REPLACES the old plan

---

## 11. Step 6 — Router + graph


In [ ]:
from langgraph.graph import StateGraph, START, END

def should_end(state: PlanState) -> str:
    """If the replanner wrote a response, we're done. Otherwise keep executing.

    Guard: `.get("response")` — the key may not exist at all on the first pass.
    """
    if state.get("response"):
        return END
    return "executor"


builder = StateGraph(PlanState)

builder.add_node("planner", plan_node)
builder.add_node("executor", execute_node)
builder.add_node("replanner", replan_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")     # always execute after planning
builder.add_edge("executor", "replanner")   # always replan after executing

# The loop: replanner -> executor  OR  replanner -> END
builder.add_conditional_edges("replanner", should_end, ["executor", END])

plan_graph = builder.compile()
print(plan_graph.get_graph().draw_mermaid())

In [ ]:
from IPython.display import Image, display
try:
    display(Image(plan_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(rendering unavailable)")

---

## 12. Step 7 — Run it


In [ ]:
goal = ("Plan a 3-night trip from Cologne to Salzburg. "
        "I need flights, a hotel, the top attractions, and the total budget.")

# recursion_limit guards against a replanner that never says "done"
final = plan_graph.invoke(
    {"input": goal, "past_steps": []},
    config={"recursion_limit": 30},
)

print("\n" + "=" * 60)
print("FINAL ANSWER:\n")
print(final["response"])

### Streaming — watch the plan shrink


In [ ]:
for chunk in plan_graph.stream(
    {"input": "Find flights Cologne to Vienna and tell me the top 2 attractions there.",
     "past_steps": []},
    config={"recursion_limit": 30},
    stream_mode="updates",
):
    for node, update in chunk.items():
        if "plan" in update:
            print(f"[{node}] plan now has {len(update['plan'])} step(s)")
        elif "past_steps" in update:
            print(f"[{node}] completed: {update['past_steps'][0][0]}")
        elif "response" in update:
            print(f"[{node}] FINAL")

---

## 13. Bonus — Human-in-the-loop on the plan

**This is the killer feature of planning agents.** Because the plan is an explicit artifact,
a human can approve or edit it *before* any money is spent.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# interrupt_after=["planner"] pauses the graph right after the plan is produced.
hitl_graph = builder.compile(
    checkpointer=InMemorySaver(),      # interrupts REQUIRE a checkpointer to save state
    interrupt_after=["planner"],
)

config = {"configurable": {"thread_id": "trip-1"}, "recursion_limit": 30}

# 1. Run -> stops after planning
hitl_graph.invoke({"input": "Plan a 2-night trip to Vienna.", "past_steps": []}, config)

# 2. Inspect the proposed plan
snapshot = hitl_graph.get_state(config)
print("\nProposed plan:", snapshot.values["plan"])
print("Paused before:", snapshot.next)

# 3. A human EDITS the plan directly in state
hitl_graph.update_state(config, {"plan": ["Find hotels in Vienna for 2 nights"]})
print("Plan overridden by human ✏️")

# 4. Resume with input=None -> "continue from where you paused"
out = hitl_graph.invoke(None, config)
print("\nResult:", out.get("response", "")[:300])

---

## 14. Cheat Sheet

```
DEFINITION   plan everything upfront, execute step-by-step, replan on new info
GRAPH SHAPE  planner -> executor ⇄ replanner -> END
COMPOSITION  Planning = Planner(LLM) + Executor(ReAct) + Replanner(LLM)
TERMINATION  replanner returns Response instead of Plan
STATE        plan REPLACED each time; past_steps ACCUMULATES (operator.add)
SUPERPOWER   the plan is inspectable -> human-in-the-loop
USE WHEN     long horizon, many steps, drift is a risk, plan needs approval
AVOID WHEN   1-3 steps (overkill) or environment is wildly unpredictable
```

**API essentials**

| Task | Code |
|---|---|
| Union output | `action: Union[Response, Plan]` |
| Force schema | `llm.with_structured_output(Act)` |
| Accumulate | `Annotated[list, operator.add]` |
| Replace | plain `List[str]` (no reducer) |
| Executor | `create_react_agent(llm, tools)` |
| Pause for human | `compile(checkpointer=..., interrupt_after=["planner"])` |
| Edit state | `graph.update_state(config, {...})` |
| Resume | `graph.invoke(None, config)` |

---

## 15. Common failure modes

| Symptom | Cause | Fix |
|---|---|---|
| Infinite loop | replanner re-emits completed steps | strengthen the "ONLY remaining steps" instruction |
| Plan too vague | steps aren't self-contained | require each step to carry its own context |
| Executor ignores the step | prompt buried the instruction | put "execute ONLY step 1" last, most salient |
| `KeyError: 'response'` | reading a key before it's written | use `state.get("response")` |
| Never terminates | replanner never emits `Response` | set `recursion_limit`, add a step-count cap |

---

## 16. Decision Tree

```
How many steps does the task need?
├── 1        ──────────────────► Reactive (Pattern 1)
├── 2–4      ──────────────────► ReAct (Pattern 2)
└── 5+
    ├── Does a human need to approve before execution?
    │   └── YES ───────────────► ✅ PLANNING (this notebook)
    ├── Does the agent lose track of the goal midway?
    │   └── YES ───────────────► ✅ PLANNING
    ├── Is output QUALITY the problem (not the path)?
    │   └── YES ───────────────► Reflective (Pattern 4)
    └── Are steps independent & parallelisable?
        └── YES ───────────────► Multi-Agent (Pattern 6)
```

---

## 17. Next

➡️ **Pattern 4 — Reflective / Learning Agents**: add a critic that judges the output and
sends it back for revision.
